# Chapter 5 — The Adjoint in Linear Algebra
## Arriola & Hyman (SIAM, 2026)

**Purpose:** Implement the adjoint method for linear systems. Verify Theorem 5.1 (adjoint = forward SA). Illustrate the LP duality connection and cost comparison.

**Key claims tested:**
- Adjoint gives identical SI as direct solve (Theorem 5.1); cost is r solves vs m solves; LP duality: A^T v = c is dual constraint

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
FULL = False
print('Chapter 5: The Adjoint in Linear Algebra')

In [ ]:
# ── Forward vs Adjoint for linear system Au = b ───────────────────────────────
# Example from §5.1: A = [[3,1],[1,2]], b depends on parameters q1, q2
# Response: J = u1 + u2 = c^T u, c = [1,1]

A = np.array([[3.0, 1.0], [1.0, 2.0]])
c = np.array([1.0, 1.0])

# Forward (FSE): one solve per parameter
b = lambda q: np.array(q)
q_nom = np.array([1.0, 1.0])
u_nom = np.linalg.solve(A, b(q_nom))
J_nom = c @ u_nom

db_dq1 = np.array([1.0, 0.0])  # db/dq1
db_dq2 = np.array([0.0, 1.0])  # db/dq2
du_dq1 = np.linalg.solve(A, db_dq1)  # FSE solve 1
du_dq2 = np.linalg.solve(A, db_dq2)  # FSE solve 2
dJ_dq1_fwd = c @ du_dq1
dJ_dq2_fwd = c @ du_dq2

# Adjoint: ONE solve regardless of number of parameters
v = np.linalg.solve(A.T, c)          # A^T v = c (one solve)
dJ_dq1_adj = v @ db_dq1              # dot product only
dJ_dq2_adj = v @ db_dq2              # dot product only

print('Forward SA (2 solves):')
print(f'  dJ/dq1 = {dJ_dq1_fwd:.6f}  dJ/dq2 = {dJ_dq2_fwd:.6f}')
print('Adjoint SA (1 solve + 2 dot products):')
print(f'  dJ/dq1 = {dJ_dq1_adj:.6f}  dJ/dq2 = {dJ_dq2_adj:.6f}')
err = max(abs(dJ_dq1_fwd-dJ_dq1_adj), abs(dJ_dq2_fwd-dJ_dq2_adj))
assert err < 1e-12, f'FAIL: err={err}'
print(f'PASS: Forward = Adjoint (Theorem 5.1), max diff = {err:.2e}')
print(f'Adjoint vector v = {v}  (encodes weighted contributions to J)')

In [ ]:
# ── Cost comparison figure ────────────────────────────────────────────────────
m_vals = np.arange(1, 51)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(m_vals, m_vals, 'c-', lw=2, label='Forward SA: m linear solves')
for r, col, ls in [(1,'steelblue','-'),(3,'coral','--'),(5,'seagreen',':')]:
    ax.axhline(r, color=col, lw=2, ls=ls, label=f'Adjoint: r={r} QOI(s)')
ax.set_xlabel('Number of POIs (m)', fontsize=12)
ax.set_ylabel('Number of linear solves', fontsize=12)
ax.set_title('Adjoint vs Forward SA — Cost Comparison\n(Theorem 5.1: same answer, lower cost)', fontsize=11)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_xlim(1,50); ax.set_ylim(0,51)
plt.tight_layout()
plt.savefig('ch05_fig_cost.pdf', dpi=150, bbox_inches='tight')
plt.show(); print('Fig 5 saved.')

In [ ]:
try:
    from google.colab import files
    files.download('ch05_fig_cost.pdf')
except ImportError:
    print('Not in Colab — figure saved locally.')